In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

df_demand = pd.read_excel("PGCB_date_power_demand.xlsx")
df_weather = pd.read_excel("weather_data.xlsx")
df_economic = pd.read_csv("economic_full_1.csv")

In [ ]:
df_weather = pd.read_excel("weather_data.xlsx", skiprows=2, header=1)

df_demand = df_demand.sort_values('datetime').reset_index(drop=True)

In [ ]:
import numpy as np

upper_limit = df_demand['demand_mw'].quantile(0.99)
lower_limit = df_demand['demand_mw'].quantile(0.01)

df_demand['demand_mw_clipped'] = df_demand['demand_mw'].clip(upper=upper_limit)

is_anomaly = (df_demand['demand_mw'] > upper_limit) | (df_demand['demand_mw'] < lower_limit)

rolling_median = df_demand['demand_mw'].rolling(window=5, center=True).median()

df_demand['demand_mw_clean'] = df_demand['demand_mw']
df_demand.loc[is_anomaly, 'demand_mw_clean'] = rolling_median

In [ ]:
# Merging demand and weather
df_weather.rename(columns={'time': 'datetime'}, inplace=True)

df_demand['datetime'] = pd.to_datetime(df_demand['datetime'])
df_weather['datetime'] = pd.to_datetime(df_weather['datetime'])

df = pd.merge(df_demand, df_weather, on='datetime', how='left')

df['demand_mw_clean'].fillna(method='bfill', inplace=True)

cols_to_drop = ['wind', 'india_adani', 'nepal', 'remarks']
df.drop(columns=cols_to_drop, inplace=True)

df.ffill(inplace=True)
df.bfill(inplace=True)


In [ ]:
# Economic data processing
df['year'] = df['datetime'].dt.year

indicators = [
    'Access to electricity (% of population)',
    'Urban population',
    'Rural population',
    'Industry (including construction), value added (% of GDP)',
    'Electric power transmission and distribution losses (% of output)',
    'GDP per unit of energy use (PPP $ per kg of oil equivalent)',
]

df_economic = df_economic[df_economic['Indicator Name'].isin(indicators)]

year_cols = [col for col in df_economic.columns if col.isdigit()]

df_economic = df_economic.melt(
    id_vars=['Indicator Name'],
    value_vars=year_cols,
    var_name='year',
    value_name='value'
)

df_economic = df_economic.pivot(
    index='year',
    columns='Indicator Name',
    values='value'
).reset_index()

df_economic['year'] = df_economic['year'].astype(int)

df = pd.merge(df, df_economic, on='year', how='left')

df.ffill(inplace=True)
df.bfill(inplace=True)


In [ ]:
# Feature engineering
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['month'] = df['datetime'].dt.month
df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)

df['lag1'] = df['demand_mw_clean'].shift(1)
df['lag24'] = df['demand_mw_clean'].shift(24)
df['lag168'] = df['demand_mw_clean'].shift(168)

df['rolling_mean_3'] = df['demand_mw_clean'].rolling(3).mean()
df['rolling_mean_24'] = df['demand_mw_clean'].rolling(24).mean()
df['std_24'] = df['demand_mw_clean'].rolling(24).std()

df = df.dropna()

df['target'] = df['demand_mw_clean'].shift(-1)

In [ ]:
import re

def clean_column_names(df):
    df.columns = [
        re.sub(r'[^A-Za-z0-9_]+', '_', col)
        for col in df.columns
    ]
    return df

df = clean_column_names(df)

In [ ]:
train = df[df['year'] < 2023]
test  = df[df['year'] == 2023]

X_train = train.drop(columns=['target', 'datetime', 'demand_mw', 'demand_mw_clipped'])
y_train = train['target']

X_test = test.drop(columns=['target', 'datetime', 'demand_mw', 'demand_mw_clipped'])
y_test = test['target']


In [ ]:
# LightGBM model
import lightgbm as lgb
from sklearn.metrics import mean_absolute_percentage_error

model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

y_pred_lgb = model.predict(X_test)
mape_lgb = mean_absolute_percentage_error(y_test, y_pred_lgb)

print("LightGBM MAPE:", mape_lgb)

lgb.plot_importance(model, max_num_features=15)
plt.show()


In [ ]:
# XGBoost model
import xgboost as xgb

model_xgb = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
model_xgb.fit(X_train, y_train)

y_pred_xgb = model_xgb.predict(X_test)
mape_xgb = mean_absolute_percentage_error(y_test, y_pred_xgb)

print("XGBoost MAPE:", mape_xgb)